# 00 · Environment setup and checks

Phase 0 of the CFFM-Net pilot. Before spending a single GPU-hour on training we prove, on this exact machine, that every moving part works: the libraries, the GPUs, the selective scan, the model, the paired data loader, two-GPU training, evaluation and the degradation probe. If something is broken this is the cheapest place to find out: everything downstream would have failed too.

| | |
|---|---|
| **Kaggle settings** | Accelerator **GPU T4 x2**, Internet **on** |
| **Inputs to attach** | `cffm-net-pilot` (the uploaded zip) |
| **Expected runtime** | about 10 to 15 minutes |
| **Produces** | a go / no-go answer, measured scan speed and memory, and `weights/yolo26n.pt` |

In [ ]:
# --------------------------------------------------------------------------------------------
# Setup: the same cell opens every notebook. It finds the code, installs it, and checks the GPU.
#
# On Kaggle the uploaded cffm-net-pilot.zip appears, already unzipped, somewhere under
# /kaggle/input. Inputs are read-only, so we copy the project to /kaggle/working and install
# it from there ("pip install -e" makes `import cffm` work in this kernel, in DataLoader
# workers, and in the extra processes Ultralytics starts when it trains on two GPUs).
# Locally, notebooks live in cffm-net-pilot/notebooks/<phase>/; the project is the first folder up
# that contains src/cffm.
# --------------------------------------------------------------------------------------------
import glob, shutil, subprocess, sys
from pathlib import Path

ON_KAGGLE = Path("/kaggle/working").exists()
if ON_KAGGLE:
    # up to five levels deep: /kaggle/input/<slug>/, /kaggle/input/<slug>/cffm-net-pilot/,
    # /kaggle/input/datasets/<owner>/<slug>/..., whichever layout this Kaggle version uses
    hits = [Path(p).parent for k in range(1, 6) for p in glob.glob("/kaggle/input" + "/*" * k + "/pyproject.toml")
            if (Path(p).parent / "src" / "cffm").is_dir()]
    assert hits, "Add the 'cffm-net-pilot' dataset (the uploaded zip) as an input to this notebook."
    PROJECT = Path("/kaggle/working/cffm-net-pilot")
    if not PROJECT.exists():
        shutil.copytree(hits[0], PROJECT, copy_function=shutil.copyfile)
        for p in [PROJECT, *PROJECT.rglob("*")]:     # the input was read-only; our copy must be writable
            p.chmod(p.stat().st_mode | 0o200)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ultralytics==8.4.171",
                    "faster-coco-eval>=1.6.7", "cloudpickle"], check=True)   # cloudpickle: two-GPU launcher
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", str(PROJECT)], check=True)
else:
    PROJECT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / "src" / "cffm").is_dir())
sys.path.insert(0, str(PROJECT / "src"))

import cffm
from cffm import env as cenv, pipeline
print("cffm", cffm.__version__, "imported from", Path(cffm.__file__).parent)   # must be inside PROJECT
env = cenv.setup()                       # prints platform, GPUs and paths; moves into the project folder
plan = pipeline.load_plan(env.project)   # configs/pilot.yaml: every run and its settings

## 0. The pilot at a glance

Eleven notebooks in two phases. Solid arrows carry data and checkpoints; dashed arrows carry metrics. The four training notebooks (04, 05, 07, 08) are independent Kaggle sessions. Notebooks 09 and 15 read their saved outputs, and 15 decides the five pre-registered hypotheses.

In [ ]:
from IPython.display import Image, display
for name in ('fig_workflow',):
    f = env.project / "docs" / "figures" / f"{name}.png"
    if f.exists():
        display(Image(filename=str(f), width=900))
    else:
        print("missing", f)

## 1. What are we running on?

In [ ]:
import json, torch
print(json.dumps(cenv.versions(), indent=2))
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f"GPU {i}: {p.name}, {p.total_memory / 2**30:.1f} GB, compute capability {p.major}.{p.minor}")
# A T4 is compute capability 7.5. It has no bfloat16, so mixed precision uses float16, which is why
# cffm/scan.py always runs the recurrence itself in float32 (half precision loses long products of
# numbers below 1 very quickly).
assert torch.cuda.is_available(), "No GPU: set Accelerator to 'GPU T4 x2' in the notebook settings."

## 2. COCO-pretrained YOLO26-n weights

Both backbones of every dual-stream model start from these weights, and the single-sensor baselines fine-tune them. We put one copy in `weights/` so every run uses the identical file.

In [ ]:
import torch
w = env.weights / "yolo26n.pt"
if not w.exists():
    torch.hub.download_url_to_file("https://github.com/ultralytics/assets/releases/download/v8.4.0/yolo26n.pt", str(w))
print(w, f"{w.stat().st_size / 1e6:.1f} MB")

## 3. (Optional) the fused Mamba kernel

The pilot does **not** need it. `cffm/scan.py` contains a pure-PyTorch selective scan that runs on any GPU
and is tested against a naive loop. The fused `mamba_ssm` kernel is faster, but it compiles from source
(20+ minutes) and may not support this PyTorch build or the T4. Leave the flag `False` unless you want to
try; if it installs, restart the kernel and the code picks it up automatically (`scan_impl: auto`).

In [ ]:
TRY_FUSED_SCAN = False
if TRY_FUSED_SCAN:
    r = subprocess.run([sys.executable, "-m", "pip", "install", "--no-build-isolation", "causal-conv1d", "mamba-ssm"],
                       capture_output=True, text=True)
    print(r.stdout[-3000:], r.stderr[-3000:])
    print("Now restart the kernel and run the notebook again from the top.")
from cffm import scan
print("fused kernel available:", scan.HAS_MAMBA_SSM)

## 4. Unit tests

Sixty-two small tests, each checking one claim the method depends on. Two need the optional fused kernel and
four are an opt-in smoke run (section 7 below does that run itself), so expect 56 passed and 6 skipped.

* the chunked scan equals the textbook recurrence, values *and* gradients;
* a token with step 0 cannot write into the state (the property reliability gating relies on);
* the scan's token order really interleaves visible and thermal, and every direction maps back exactly;
* a fresh CMFM block is exactly a reliability-weighted average (its output projection starts at zero);
* every model config builds and runs, and COCO weights fill **both** backbones;
* paired loading returns 4 registered channels;
* the Phase 2 Temporal State Memory holds, resets and warps as specified.

In [ ]:
r = subprocess.run([sys.executable, "-m", "pytest", "-q", "tests"], cwd=env.project, capture_output=True, text=True)
print(r.stdout[-4000:])
assert r.returncode == 0, r.stderr[-4000:]

## 5. How fast is the scan here?

The biggest scan in CFFM-Net sits at stride 8. A 640 x 640 training crop gives an 80 x 80 map; interleaving
visible and thermal doubles it to **12 800 tokens**; four directions of 64 channels make **256 scan
channels**; and Kaggle trains **8 images per GPU**. That is the size we time below, forward and backward.

In [ ]:
import time, torch
from cffm.scan import selective_scan

def time_scan(b=8, d=256, L=12800, n=8, groups=4, impl="torch", reps=3):
    dev = "cuda:0"
    u = torch.randn(b, d, L, device=dev, requires_grad=True)
    delta = torch.nn.functional.softplus(torch.randn(b, d, L, device=dev))
    A = -torch.exp(torch.randn(d, n, device=dev))
    B, C = torch.randn(b, groups, n, L, device=dev), torch.randn(b, groups, n, L, device=dev)
    D = torch.ones(d, device=dev)
    torch.cuda.reset_peak_memory_stats(); times = []
    for _ in range(reps + 1):
        torch.cuda.synchronize(); t = time.time()
        y = selective_scan(u, delta, A, B, C, D, impl=impl, **({"use_checkpoint": True} if impl == "torch" else {}))
        y.sum().backward(); torch.cuda.synchronize(); times.append(time.time() - t)
    return 1000 * sorted(times[1:])[len(times[1:]) // 2], torch.cuda.max_memory_allocated() / 2**30

ms, gb = time_scan()
print(f"PyTorch scan, stride-8 size, batch 8: {ms:.0f} ms forward+backward, peak {gb:.2f} GB")
if scan.HAS_MAMBA_SSM:
    ms, gb = time_scan(impl="cuda")
    print(f"fused kernel, same size:            {ms:.0f} ms forward+backward, peak {gb:.2f} GB")

## 6. Do the planned batch sizes fit?

One forward and backward pass per model at its planned batch per GPU (from `configs/pilot.yaml`), with mixed
precision and random inputs. This is not training: it only measures memory and step time, so we know before
the real runs whether a batch size has to come down. The last column projects one 30-epoch run on LLVIP's
~4 000 training pairs.

In [ ]:
import time, torch, pandas as pd
from ultralytics.nn.tasks import DetectionModel
from cffm.model import DualStreamDetectionModel

n_gpu = max(1, torch.cuda.device_count())
def tensors(o):
    """Every tensor inside nested dicts / lists / tuples (the head's training output is nested)."""
    if torch.is_tensor(o): return [o]
    if isinstance(o, dict): o = list(o.values())
    return [t for x in o for t in tensors(x)] if isinstance(o, (list, tuple)) else []

def step(model, b, ch, hw=(640, 640), reps=3):
    m = model.to("cuda:0").train(); opt = torch.optim.SGD(m.parameters(), lr=1e-4)
    x = torch.rand(b, ch, *hw, device="cuda:0")
    torch.cuda.reset_peak_memory_stats(); times = []
    for _ in range(reps + 1):
        torch.cuda.synchronize(); t = time.time()
        with torch.autocast("cuda", dtype=torch.float16):
            loss = sum(t.float().mean() for t in tensors(m(x)) if t.requires_grad)   # a stand-in loss
        loss.backward(); opt.step(); opt.zero_grad(set_to_none=True)
        torch.cuda.synchronize(); times.append(time.time() - t)
    return sorted(times[1:])[len(times[1:]) // 2], torch.cuda.max_memory_allocated() / 2**30

rows, seen = [], set()
for s in (pipeline.run_spec(plan, r["name"]) for r in plan["runs"]):
    if s["model"] in seen:
        continue
    seen.add(s["model"])
    dual = s["model"].startswith("configs/")
    model = DualStreamDetectionModel(str(env.project / s["model"]), nc=1, verbose=False) if dual \
        else DetectionModel(s["model"], nc=1, verbose=False)
    per_gpu = s["batch"] // n_gpu
    t, gb = step(model, per_gpu, 4 if dual else 3)
    hours = 4000 / s["batch"] * t * s["epochs"] / 3600 * 1.15      # +15% for data loading and DDP sync
    rows.append({"model": Path(s["model"]).stem, "batch/GPU": per_gpu, "peak GB": round(gb, 2),
                 "step s": round(t, 3), "~hours for 30 ep on LLVIP": round(hours, 2)})
    del model; torch.cuda.empty_cache()
df = pd.DataFrame(rows); print(df.to_string(index=False))
limit = min(torch.cuda.get_device_properties(i).total_memory for i in range(n_gpu)) / 2**30
assert (df["peak GB"] < limit - 1.0).all(), "A batch is too large for this GPU: lower it in configs/pilot.yaml."

## 7. The whole pipeline, end to end, on fake data and both GPUs

A tiny synthetic paired dataset (bright squares that are hot in thermal), one epoch per model, with the
real trainer, the two-GPU (DDP) launcher, the validator with size-binned COCO metrics, and the degradation
probe. The numbers mean nothing; the point is that no step crashes on this machine.

In [ ]:
from cffm.data import make_synthetic
from cffm.train import evaluate, probe, train_run

synth = make_synthetic(env.data / "synthetic", n_train=32, n_val=8, imgsz=320)
smoke = {}
for name, model, data in [("smoke_cffm", str(env.project / "configs/models/cffm-net-n.yaml"), "data_paired.yaml"),
                          ("smoke_concat", str(env.project / "configs/models/two-stream-concat-n.yaml"), "data_paired.yaml"),
                          ("smoke_visible", "yolo26n.yaml", "data_visible.yaml")]:
    best = train_run(name, model, str(synth / data), pretrained=pipeline.weights_file(env), epochs=1, imgsz=320,
                     batch=8, device=env.device, project=str(env.runs / "smoke"), workers=2, plots=False)
    smoke[name] = evaluate(best, synth / data, imgsz=320, batch=8, device=0, project=str(env.runs / "smoke_eval"))
    print(name, "ok:", {k: round(v, 3) for k, v in smoke[name].items() if k in ("mAP50(B)", "AP_s(B)")})

best = env.runs / "smoke" / "smoke_cffm" / "weights" / "best.pt"
for kind, flagged in [("clean", False), ("thermal_drop", False), ("thermal_drop", True), ("thermal_shift_8", False)]:
    m = probe(best, synth / "data_paired.yaml", kind, flagged=flagged, imgsz=320, batch=8, device=0,
              project=str(env.runs / "smoke_probe"))
    print("probe", kind, "flagged" if flagged else "", "ok")

## 8. Verdict

If every cell above ran without an error, the environment is ready and the code is verified on this
hardware. Clean up the smoke runs (they are not results), then continue with
**01 · Data acquisition**.

In [ ]:
shutil.rmtree(env.runs / "smoke", ignore_errors=True)
for d in ("smoke_eval", "smoke_probe"):
    shutil.rmtree(env.runs / d, ignore_errors=True)
shutil.rmtree(env.data / "synthetic", ignore_errors=True)
print("Phase 0 checks passed. Next: 01_data_acquisition.ipynb")